In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from datetime import datetime
from scipy.stats import pearsonr
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
import pingouin as pg

# ==============================================================================
# CONFIGURATION AND DATA STRUCTURES
# ==============================================================================

@dataclass
class AnalysisConfig:
    """Centralized configuration for all analysis parameters."""
    base_output_dir: Path = Path('motor_learning_output')
    min_complete_strides: int = 20
    motor_noise_strides: int = 20
    motor_noise_threshold: float = 0.3
    success_rate_threshold: float = 0.68
    target_size_threshold: float = 0.31
    max_strides_threshold: int = 415
    figure_dpi: int = 300
    alpha_level: float = 0.05
    age_bins: List[int] = field(default_factory=lambda: [7, 10, 13, 16, 18])
    age_labels: List[str] = field(default_factory=lambda: ['7-10', '10-13', '13-16', '16-18'])
    trial_type_mapping: Dict[str, str] = field(default_factory=lambda: {
        'primer': 'vis1', 'trial': 'invis', 'vis': 'vis2', 'pref': 'pref'
    })
    
    def __post_init__(self):
        dirs = ['figures', 'individual_plots', 'population_plots', 'statistical_plots', 
                'reports', 'exports', 'processed_data']
        for d in dirs:
            setattr(self, f'{d.replace("_", "")}_dir', self.base_output_dir / d)
            getattr(self, f'{d.replace("_", "")}_dir').mkdir(parents=True, exist_ok=True)
        
        for subdir in ['individual_plots', 'population_plots', 'statistical_plots']:
            (self.base_output_dir / 'figures' / subdir).mkdir(parents=True, exist_ok=True)
        
        self.processed_data_file = self.processeddata_dir / 'processed_data.pkl'

@dataclass
class SubjectData:
    subject_id: str
    metadata: Dict[str, Any]
    trial_data: Dict[str, Dict[str, Any]]
    
    @property
    def age(self) -> float:
        return self.metadata.get('age_months', np.nan) / 12

# ==============================================================================
# BASE CLASSES
# ==============================================================================

class BaseProcessor(ABC):
    def __init__(self, config: AnalysisConfig, debug: bool = True):
        self.config = config
        self.debug = debug
    
    def log(self, message: str, level: str = "info"):
        if self.debug:
            symbols = {"info": "📊", "warning": "⚠️", "error": "❌", "success": "✓"}
            print(f"{symbols.get(level, '•')} {message}")

class BaseVisualizer(ABC):
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.colors = {
            'primary': '#667eea', 'secondary': '#764ba2', 'success': '#28a745',
            'warning': '#ffc107', 'danger': '#dc3545', 'vis1': '#1f77b4',
            'invis': '#ff7f0e', 'vis2': '#2ca02c'
        }
    
    def save_figure(self, fig: plt.Figure, filename: str, subdir: str = 'general'):
        subdir_map = {
            'individual': self.config.base_output_dir / 'figures' / 'individual_plots',
            'population': self.config.base_output_dir / 'figures' / 'population_plots',
            'statistical': self.config.base_output_dir / 'figures' / 'statistical_plots'
        }
        save_path = subdir_map.get(subdir, self.config.figures_dir) / filename
        fig.savefig(save_path, dpi=self.config.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        return save_path
    
    def add_trendline(self, ax, x, y):
        x_clean, y_clean = pd.to_numeric(x, errors='coerce'), pd.to_numeric(y, errors='coerce')
        valid = x_clean.notna() & y_clean.notna()
        if valid.sum() < 2:
            return
        
        x_vals, y_vals = x_clean[valid], y_clean[valid]
        try:
            coeffs = np.polyfit(x_vals, y_vals, 1)
            trendline = np.poly1d(coeffs)
            r, p = pearsonr(x_vals, y_vals)
            ax.plot(x_vals, trendline(x_vals), 'r--', alpha=0.8, linewidth=2)
            ax.text(0.05, 0.95, f'r² = {r**2:.3f}\np = {p:.3f}\nn = {len(x_vals)}', 
                   transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top', fontsize=10)
        except Exception:
            pass

# ==============================================================================
# DATA PROCESSING COMPONENTS
# ==============================================================================

class DataValidator:
    @staticmethod
    def validate_dataframe(df: pd.DataFrame, required_cols: List[str] = None) -> bool:
        return df is not None and not df.empty and (not required_cols or all(col in df.columns for col in required_cols))
    
    @staticmethod
    def detect_anomalies(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        if df is None or df.empty:
            return df, {}
        
        df = df.copy()
        df['Anomalous'] = False
        anomalies = {}
        
        # Time-based anomalies
        time_col = next((col for col in ['Time', 'Timestamp', 'Time (s)'] if col in df.columns), None)
        if time_col:
            df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
            time_diff = df[time_col].diff()
            jump_mask = time_diff > time_diff.quantile(0.99) * 5
            for idx in df.index[jump_mask.fillna(False)]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('time_jump')
        
        # Sum of gains and steps anomalies
        if 'Sum of gains and steps' in df.columns:
            high_mask = df['Sum of gains and steps'] > 4
            zero_mask = df['Sum of gains and steps'] == 0
            for idx in df.index[high_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_high')
            for idx in df.index[zero_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_zero')
        
        return df, anomalies

class FileLoader:
    @staticmethod
    def load_file(file_path: Path) -> Optional[pd.DataFrame]:
        try:
            df = pd.read_csv(file_path, sep='\t')
            if 'Stride Number' in df.columns:
                df['Stride Number'] = pd.to_numeric(df['Stride Number'], errors='coerce')
                df = df.dropna(subset=['Stride Number']).drop_duplicates(subset=['Stride Number']).sort_values('Stride Number')
            return df if not df.empty else None
        except Exception:
            return None

class TrialProcessor(BaseProcessor):
    def process_trial_files(self, subject_dir: Path, trial_prefix: str) -> Optional[pd.DataFrame]:
        all_files = sorted(subject_dir.glob(f"{trial_prefix}*.txt"))
        if not all_files:
            return None
        
        if trial_prefix == 'pref':
            return FileLoader.load_file(max(all_files, key=lambda f: f.stat().st_size))
        
        if len(all_files) == 1:
            return FileLoader.load_file(all_files[0])
        
        return self._combine_trial_fragments(all_files)
    
    def _combine_trial_fragments(self, files: List[Path]) -> Optional[pd.DataFrame]:
        dfs = [FileLoader.load_file(f) for f in files if FileLoader.load_file(f) is not None]
        if not dfs:
            return None
        
        combined = pd.concat(dfs, ignore_index=True)
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number').drop_duplicates('Stride Number')
        return combined

# ==============================================================================
# METRICS CALCULATION
# ==============================================================================

class MetricsEngine:
    def __init__(self, config: AnalysisConfig):
        self.config = config
    
    def calculate_period_metrics(self, period_data: pd.DataFrame, trial_type: str, condition: str) -> Dict:
        metrics = {f'{trial_type}_sr_{condition}_const': period_data['Success'].mean()}
        
        if 'Sum of gains and steps' in period_data.columns:
            sogs = period_data['Sum of gains and steps']
            metrics.update({
                f'{trial_type}_sd_{condition}_const': sogs.std(),
                f'{trial_type}_msl_{condition}_const': sogs.mean()
            })
            if 'Constant' in period_data.columns:
                metrics[f'{trial_type}_error_{condition}_const'] = (sogs - period_data['Constant']).mean()
        
        if all(col in period_data.columns for col in ['Right step length', 'Left step length']):
            asymmetry = self._calculate_asymmetry(period_data['Right step length'], period_data['Left step length'])
            if asymmetry is not None:
                metrics[f'{trial_type}_asymmetry_{condition}_const'] = asymmetry
        
        strides_between = self._calculate_strides_between_successes(period_data)
        if strides_between is not None:
            metrics[f'{trial_type}_strides_between_success_{condition}_const'] = strides_between
        
        return metrics
    
    def calculate_preference_metrics(self, pref_df: pd.DataFrame) -> Dict:
        metrics = {'mot_noise': None, 'pref_asymmetry': None}
        
        if pref_df is None or pref_df.empty or not all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
            return metrics
        
        right_steps = pref_df['Right step length']
        left_steps = pref_df['Left step length']
        right_clean = right_steps[(right_steps != 0) & (right_steps.notna())]
        left_clean = left_steps[(left_steps != 0) & (left_steps.notna())]
        
        min_required = self.config.motor_noise_strides
        if len(right_clean) < min_required or len(left_clean) < min_required:
            return metrics
        
        final_right, final_left = right_clean.iloc[-1], left_clean.iloc[-1]
        if final_right <= 0 or final_left <= 0:
            return metrics
        
        norm_right, norm_left = right_clean / final_right, left_clean / final_left
        min_length = min(len(norm_right), len(norm_left))
        if min_length < min_required:
            return metrics
        
        sum_steps = norm_right.iloc[:min_length] + norm_left.iloc[:min_length]
        
        if len(sum_steps) >= self.config.motor_noise_strides:
            noise = sum_steps.tail(self.config.motor_noise_strides).std()
            if not pd.isna(noise) and noise > 0:
                metrics['mot_noise'] = noise
        
        if len(right_clean) >= self.config.motor_noise_strides and len(left_clean) >= self.config.motor_noise_strides:
            last_n_right = right_clean.tail(self.config.motor_noise_strides) / final_right
            last_n_left = left_clean.tail(self.config.motor_noise_strides) / final_left
            asymmetry = self._calculate_asymmetry(last_n_right.values, last_n_left.values)
            if asymmetry is not None:
                metrics['pref_asymmetry'] = asymmetry
        
        return metrics
    
    def _calculate_asymmetry(self, right_values, left_values) -> Optional[float]:
        denominator = right_values + left_values
        valid_mask = denominator != 0
        if not valid_mask.any():
            return None
        asymmetry_vals = np.abs((right_values - left_values) / denominator)[valid_mask]
        return np.mean(asymmetry_vals) if len(asymmetry_vals) > 0 else None
    
    def _calculate_strides_between_successes(self, df: pd.DataFrame) -> Optional[float]:
        if df is None or 'Success' not in df.columns:
            return None
        df = df.reset_index(drop=True)
        success_positions = df.index[df['Success'] == 1].tolist()
        return np.mean(np.diff(success_positions)) if len(success_positions) >= 2 else None

# ==============================================================================
# DATA MANAGEMENT
# ==============================================================================

class DataManager(BaseProcessor):
    def __init__(self, metadata_path: str, data_root_dir: str, config: AnalysisConfig, 
                 force_reprocess: bool = False, debug: bool = True):
        super().__init__(config, debug)
        self.metadata_path = metadata_path
        self.data_root_dir = Path(data_root_dir)
        self.trial_processor = TrialProcessor(config, debug)
        self.metrics_engine = MetricsEngine(config)
        self.subjects: Dict[str, SubjectData] = {}
        self.metadata: pd.DataFrame = None
        
        if not force_reprocess and config.processed_data_file.exists():
            self._load_processed_data()
        else:
            self._process_all_data()
            self._save_processed_data()
    
    def _load_processed_data(self):
        try:
            with open(self.config.processed_data_file, 'rb') as f:
                saved_data = pickle.load(f)
            
            for subject_id, data in saved_data.items():
                self.subjects[subject_id] = SubjectData(
                    subject_id=subject_id, metadata=data['metadata'], trial_data=data['trial_data']
                )
            
            self.metadata = pd.DataFrame.from_dict(
                {subj: data.metadata for subj, data in self.subjects.items()}, orient='index'
            )
            self.log(f"Loaded {len(self.subjects)} subjects from cache", "success")
        except Exception as e:
            self.log(f"Failed to load cached data: {e}", "warning")
            self._process_all_data()
            self._save_processed_data()
    
    def _save_processed_data(self):
        try:
            save_data = {
                subject_id: {'metadata': subject.metadata, 'trial_data': subject.trial_data}
                for subject_id, subject in self.subjects.items()
            }
            with open(self.config.processed_data_file, 'wb') as f:
                pickle.dump(save_data, f)
            self.log(f"Saved processed data to {self.config.processed_data_file}", "success")
        except Exception as e:
            self.log(f"Failed to save processed data: {e}", "error")
    
    def _process_all_data(self):
        self._load_metadata()
        total_subjects = len(self.metadata)
        self.log(f"Processing {total_subjects} subjects...")
        
        for i, (_, row) in enumerate(self.metadata.iterrows(), 1):
            if self.debug and i % 10 == 0:
                self.log(f"Progress: {i}/{total_subjects}")
            
            subject_data = self._process_subject(row['ID'], row)
            if subject_data:
                self.subjects[row['ID']] = subject_data
    
    def _load_metadata(self):
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata['DOB'] = pd.to_datetime(self.metadata['DOB'], errors='coerce')
        self.metadata['Session Date'] = pd.to_datetime(self.metadata['Session Date'], errors='coerce')
        self.metadata = self.metadata.dropna(subset=['ID', 'age_months'])
    
    def _process_subject(self, subject_id: str, metadata_row: pd.Series) -> Optional[SubjectData]:
        subject_dir = self.data_root_dir / subject_id
        if not subject_dir.exists():
            return None
        
        trial_data = {}
        for original_type, mapped_type in self.config.trial_type_mapping.items():
            df = self.trial_processor.process_trial_files(subject_dir, original_type)
            if df is not None:
                if original_type != 'pref':
                    df, anomalies = self._process_trial_data(df)
                else:
                    df = df.drop_duplicates(subset='Left heel strike', keep='last')
                    anomalies = {}
                trial_data[mapped_type] = {'data': df, 'anomalies': anomalies}
        
        return SubjectData(subject_id=subject_id, metadata=metadata_row.to_dict(), trial_data=trial_data) if trial_data else None
    
    def _process_trial_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        if df is None or df.empty:
            return None, {}
        
        required_cols = ['Stride Number', 'Success', 'Upper bound success', 'Lower bound success', 'Constant']
        if not all(col in df.columns for col in required_cols):
            return None, {}
        
        df = df.sort_values('Stride Number')
        df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        df = df.drop_duplicates(subset='Stride Number', keep='last')
        
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        return DataValidator.detect_anomalies(df)
    
    def calculate_metrics(self) -> pd.DataFrame:
        results = []
        self.log(f"Calculating metrics for {len(self.subjects)} subjects...")
        
        for subject_id, subject in self.subjects.items():
            result = self._calculate_subject_metrics(subject)
            if result:
                results.append(result)
        
        if not results:
            self.log("No valid metrics calculated!", level="error")
            return pd.DataFrame()
        
        df = pd.DataFrame(results).infer_objects()
        self.log(f"Successfully calculated metrics for {len(df)} subjects", level="success")
        return df
    
    def _calculate_subject_metrics(self, subject: SubjectData) -> Optional[Dict]:
        result = {
            'ID': subject.subject_id,
            'age': subject.age,
            'session_date': subject.metadata.get('Session Date')
        }
        
        for trial_type in ['vis1', 'invis', 'vis2']:
            trial_dict = subject.trial_data.get(trial_type)
            if not trial_dict or trial_dict['data'] is None or trial_dict['data'].empty or 'Success' not in trial_dict['data'].columns:
                continue
            
            df = trial_dict['data']
            
            for condition in ['max', 'min']:
                period_data, indices = self._get_period_data(df, condition)
                if period_data is not None and not period_data.empty:
                    metrics = self.metrics_engine.calculate_period_metrics(period_data, trial_type, condition)
                    result.update(metrics)
                    result[f'{trial_type}_{condition}_const_indices'] = indices
            
            if df is not None:
                result.update({
                    f'{trial_type}_min_target_size': df['Target size'].min() if 'Target size' in df.columns else None,
                    f'{trial_type}_max_constant': df['Constant'].max() if 'Constant' in df.columns else None,
                    f'{trial_type}_min_constant': df['Constant'].min() if 'Constant' in df.columns else None
                })
                
                if trial_type == 'invis':
                    result.update(self._calculate_condition_order(df))
        
        pref_dict = subject.trial_data.get('pref')
        if pref_dict:
            pref_metrics = self.metrics_engine.calculate_preference_metrics(pref_dict['data'])
            result.update(pref_metrics)
        
        return result
    
    def _get_period_data(self, df: pd.DataFrame, condition: str, length: int = 20) -> Tuple[Optional[pd.DataFrame], Optional[List]]:
        if df is None or df.empty or not all(col in df.columns for col in ['Target size', 'Constant']):
            return None, None
        
        min_target = df['Target size'].min()
        min_target_periods = df[df['Target size'] <= min_target + 0.001]
        if min_target_periods.empty:
            return None, None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max' 
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)]
        return (period_data.tail(length), period_data.index.tolist()) if not period_data.empty else (None, None)
    
    def _calculate_condition_order(self, df: pd.DataFrame) -> Dict:
        all_max_indices = df.index[df['Constant'] == df['Constant'].max()].tolist()
        all_min_indices = df.index[df['Constant'] == df['Constant'].min()].tolist()
        
        if all_max_indices and all_min_indices:
            first_max, first_min = min(all_max_indices), min(all_min_indices)
            return {'invis_max_first': first_max < first_min, 'invis_min_first': first_min < first_max}
        return {'invis_max_first': False, 'invis_min_first': False}
    
    def filter_subjects(self, max_target_size: float = None, min_age: float = None, 
                       max_age: float = None, required_trial_types: List[str] = None) -> 'DataManager':
        filtered_subjects = {}
        
        for subject_id, subject in self.subjects.items():
            age = subject.age
            if (min_age is not None and age < min_age) or (max_age is not None and age > max_age):
                continue
            
            if required_trial_types:
                missing_trials = [t for t in required_trial_types 
                                if t not in subject.trial_data or subject.trial_data[t]['data'] is None]
                if missing_trials:
                    continue
            
            valid = True
            for trial_type, trial_dict in subject.trial_data.items():
                if trial_dict and trial_dict['data'] is not None:
                    df = trial_dict['data']
                    if (max_target_size is not None and 'Target size' in df.columns and 
                        df['Target size'].min() > max_target_size):
                        valid = False
                        break
            
            if valid:
                filtered_subjects[subject_id] = subject
        
        new_manager = DataManager.__new__(DataManager)
        for attr in ['config', 'metadata_path', 'data_root_dir', 'debug', 'trial_processor', 'metrics_engine']:
            setattr(new_manager, attr, getattr(self, attr))
        new_manager.subjects = filtered_subjects
        new_manager.metadata = pd.DataFrame.from_dict(
            {subj: data.metadata for subj, data in filtered_subjects.items()}, orient='index'
        )
        return new_manager

# ==============================================================================
# STATISTICAL ANALYSIS
# ==============================================================================

class StatisticalAnalyzer:
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        self.config = config
        self.metrics_df = metrics_df
        self.filtered_df = (metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold] 
                           if 'mot_noise' in metrics_df.columns else metrics_df)

    def _run_repeated_measures_anova_generic(self, metric_type: str) -> Dict:
        """Generic ANOVA runner for different metrics (sr, msl, sd)"""
        metric_names = {'sr': 'success_rate', 'msl': 'mean_stride_length', 'sd': 'stride_length_variability'}
        dv_name = metric_names[metric_type]
        
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if f'_{metric_type}_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type, condition = parts[0], parts[2]
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id, 'trial_type': trial_type, 'condition': condition,
                                dv_name: row[col], 'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna()
        if df_long.empty:
            return {'error': f'No valid {dv_name} data for analysis'}
        
        results = self._run_basic_anova(df_long, dv_name)
        results.update(self._run_covariate_analysis(df_long, dv_name))
        results['descriptive_stats'] = self._get_descriptive_stats(df_long, dv_name)
        return results
    
    def _run_basic_anova(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Run basic repeated measures ANOVA"""
        results = {}
        try:
            # Ensure data is properly formatted for pingouin
            df_long = df_long.copy()
            df_long['subject'] = df_long['subject'].astype(str)
            df_long['trial_type'] = df_long['trial_type'].astype('category')
            df_long['condition'] = df_long['condition'].astype('category')
            
            # Main effects and interaction
            for effect, within_vars in [('trial_type', 'trial_type'), ('condition', 'condition'), 
                                       ('interaction', ['trial_type', 'condition'])]:
                if (effect != 'interaction' and len(df_long[within_vars].unique()) > 1) or \
                   (effect == 'interaction' and len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1):
                    
                    # Use more robust ANOVA settings
                    aov = pg.rm_anova(data=df_long, dv=dv_name, within=within_vars, 
                                     subject='subject', detailed=True, effsize='ng2')
                    
                    if effect == 'interaction':
                        interaction_row = aov[aov['Source'].str.contains('trial_type \\* condition')]
                        if not interaction_row.empty:
                            results[f'{effect}_effect'] = self._extract_anova_results(interaction_row.iloc[0])
                    else:
                        results[f'{effect}_effect'] = self._extract_anova_results(aov.iloc[0])
        except Exception as e:
            results['basic_anova_error'] = str(e)
        return results
    
    def _extract_anova_results(self, anova_row) -> Dict:
        """Extract standardized results from ANOVA row"""
        return {
            'F': float(anova_row['F']),
            'p_value': float(anova_row['p-unc']),
            'effect_size': float(anova_row['ng2'])
        }
    
    def _run_covariate_analysis(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Run covariate analysis"""
        results = {}
        try:
            available_covariates = [cov for cov in ['age', 'motor_noise', 'pref_asymmetry'] 
                                  if cov in df_long.columns and df_long[cov].notna().sum() > 0]
            
            if available_covariates:
                results['ancova_covariates'] = available_covariates
                
                # Covariate correlations
                for cov in available_covariates:
                    cov_corr = df_long.groupby('subject').agg({dv_name: 'mean', cov: 'first'}).reset_index()
                    if len(cov_corr) > 5:
                        r, p = pearsonr(cov_corr[cov], cov_corr[dv_name])
                        results[f'{cov}_covariate_effect'] = {
                            'correlation': float(r), 'p_value': float(p),
                            'interpretation': f'{cov} effect on {dv_name}'
                        }
                
                # Mixed effects analysis
                try:
                    import statsmodels.api as sm
                    from statsmodels.formula.api import mixedlm
                    
                    covariate_terms = ' + '.join(available_covariates)
                    formula = f"{dv_name} ~ trial_type * condition + {covariate_terms}"
                    
                    # Improve convergence by using REML and better starting values
                    model = mixedlm(formula, df_long, groups=df_long['subject'], 
                                   re_formula="1")  # Simple random intercept
                    
                    # Fit with more robust settings
                    try:
                        fitted_model = model.fit(reml=True, maxiter=200)
                    except:
                        # Fallback: try with simpler model if convergence fails
                        simple_formula = f"{dv_name} ~ trial_type + condition + {covariate_terms}"
                        model = mixedlm(simple_formula, df_long, groups=df_long['subject'])
                        fitted_model = model.fit(reml=True, maxiter=100)
                    
                    covariate_effects = {}
                    for cov in available_covariates:
                        if cov in fitted_model.params.index:
                            covariate_effects[f'{cov}_effect'] = {
                                'coefficient': float(fitted_model.params[cov]),
                                'p_value': float(fitted_model.pvalues[cov]),
                                'confidence_interval': [float(fitted_model.conf_int().loc[cov, 0]), 
                                                      float(fitted_model.conf_int().loc[cov, 1])],
                                'interpretation': f'Change in {dv_name} per unit increase in {cov}'
                            }
                    results['mixed_effects_covariates'] = covariate_effects
                    
                except Exception as e:
                    results['mixed_effects_error'] = str(e)
            
        except Exception as e:
            results['ancova_error'] = str(e)
        return results
    
    def _get_descriptive_stats(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Get descriptive statistics"""
        return {
            'n_subjects': len(df_long['subject'].unique()),
            'n_observations': len(df_long),
            f'{dv_name}_overall': float(df_long[dv_name].mean()),
            f'std_{dv_name}_overall': float(df_long[dv_name].std()),
            'by_trial_type': df_long.groupby('trial_type')[dv_name].agg(['mean', 'std', 'count']).to_dict(),
            'by_condition': df_long.groupby('condition')[dv_name].agg(['mean', 'std', 'count']).to_dict()
        }

    def run_repeated_measures_anova(self) -> Dict:
        """Run repeated measures ANOVA for success rates"""
        return self._run_repeated_measures_anova_generic('sr')
    
    def run_repeated_measures_anova_msl(self) -> Dict:
        """Run repeated measures ANOVA for mean stride length"""
        return self._run_repeated_measures_anova_generic('msl')
    
    def run_repeated_measures_anova_sd(self) -> Dict:
        """Run repeated measures ANOVA for stride length variability"""
        return self._run_repeated_measures_anova_generic('sd')

    def run_age_stratified_anova(self, age_groups: Dict[str, List[float]] = None, 
                                min_subjects_per_group: int = 1) -> Dict:
        """Run ANOVA analysis stratified by age groups"""
        if age_groups is None:
            age_groups = {'younger': [7, 12], 'middle': [12, 15], 'older': [15, 18]}
        
        # Prepare long-format data
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id, age = row.name, row.get('age', np.nan)
            if pd.isna(age):
                continue
                
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type, condition = parts[0], parts[2]
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id, 'trial_type': trial_type, 'condition': condition,
                                'success_rate': row[col], 'age': age,
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['age', 'success_rate'])
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {
            'age_group_definitions': age_groups,
            'min_subjects_threshold': min_subjects_per_group,
            'total_subjects_analyzed': len(df_long['subject'].unique()),
            'age_range': [float(df_long['age'].min()), float(df_long['age'].max())],
            'group_analyses': {}, 'between_group_comparisons': {}, 'summary_comparison': {}
        }
        
        # Assign subjects to age groups
        df_long['age_group'] = None
        group_assignments = {}
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_long['age'] >= min_age) & (df_long['age'] < max_age)
            df_long.loc[mask, 'age_group'] = group_name
            
            subjects_in_group = df_long[mask]['subject'].unique()
            group_assignments[group_name] = {
                'subjects': list(subjects_in_group), 'n_subjects': len(subjects_in_group),
                'age_range': [float(df_long[mask]['age'].min()) if len(subjects_in_group) > 0 else np.nan,
                             float(df_long[mask]['age'].max()) if len(subjects_in_group) > 0 else np.nan],
                'mean_age': float(df_long[mask]['age'].mean()) if len(subjects_in_group) > 0 else np.nan
            }
        
        results['group_assignments'] = group_assignments
        
        # Run ANOVA for each age group
        for group_name, group_info in group_assignments.items():
            if group_info['n_subjects'] < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={group_info["n_subjects"]}, minimum={min_subjects_per_group})'
                }
                continue
            
            group_data = df_long[df_long['age_group'] == group_name].copy()
            group_results = {}
            
            # Ensure proper data types
            group_data['subject'] = group_data['subject'].astype(str)
            group_data['trial_type'] = group_data['trial_type'].astype('category')
            group_data['condition'] = group_data['condition'].astype('category')
            
            try:
                aov_trial = pg.rm_anova(data=group_data, dv='success_rate', 
                                       within='trial_type', subject='subject', 
                                       detailed=True, effsize='ng2')
                group_results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size_eta2': float(aov_trial['ng2'].iloc[0]),
                    'significant': float(aov_trial['p-unc'].iloc[0]) < 0.05
                }
            except Exception as e:
                group_results['anova_error'] = str(e)
            
            # Post-hoc pairwise comparisons
            try:
                if len(group_data['trial_type'].unique()) > 2:
                    subject_means = group_data.groupby(['subject', 'trial_type'])['success_rate'].mean().reset_index()
                    subject_means['subject'] = subject_means['subject'].astype(str)
                    subject_means['trial_type'] = subject_means['trial_type'].astype('category')
                    
                    # Use the modern pairwise_tests function
                    posthoc = pg.pairwise_tests(data=subject_means, dv='success_rate', 
                                              within='trial_type', subject='subject', 
                                              padjust='bonf', effsize='hedges')
                    
                    group_results['posthoc_comparisons'] = {}
                    for _, row in posthoc.iterrows():
                        comparison = f"{row['A']}_vs_{row['B']}"
                        group_results['posthoc_comparisons'][comparison] = {
                            'mean_diff': float(row.get('mean(A)', 0) - row.get('mean(B)', 0)) if 'mean(A)' in row else 0.0,
                            't_stat': float(row['T']), 'p_corrected': float(row['p-corr']),
                            'cohens_d': float(row.get('hedges', 0.0)), 'significant': row['p-corr'] < 0.05,
                            'effect_size_interpretation': self._interpret_cohens_d(float(row.get('hedges', 0.0)))
                        }
            except Exception as e:
                group_results['posthoc_error'] = str(e)
            
            # Descriptive statistics
            try:
                group_results['mean_success_rates'] = {
                    trial: float(group_data[group_data['trial_type'] == trial]['success_rate'].mean())
                    for trial in group_data['trial_type'].unique()
                }
                group_results['descriptive_stats'] = {
                    'n_subjects': group_info['n_subjects'], 'n_observations': len(group_data),
                    'age_info': {'mean_age': group_info['mean_age'], 'age_range': group_info['age_range']}
                }
            except Exception as e:
                group_results['descriptive_error'] = str(e)
            
            results['group_analyses'][group_name] = group_results
        
        return results
    
    def _interpret_cohens_d(self, d):
        """Interpret Cohen's d effect size"""
        abs_d = abs(d)
        if abs_d < 0.2: return "negligible"
        elif abs_d < 0.5: return "small"
        elif abs_d < 0.8: return "medium"
        else: return "large"

    def run_regression_analysis(self, trial_type: str = 'invis', condition: str = 'max',
                               predictors: List[str] = None) -> Dict:
        """Run regression analysis"""
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
        
        available_predictors = [p for p in predictors if p in self.filtered_df.columns]
        target_col = f'{trial_type}_sr_{condition}_const'
        
        if target_col not in self.filtered_df.columns:
            raise ValueError(f"Target column {target_col} not found")
        
        valid_data = self.filtered_df[available_predictors + [target_col]].dropna()
        if len(valid_data) < 10:
            raise ValueError(f"Insufficient data: only {len(valid_data)} valid samples")
        
        X, y = valid_data[available_predictors], valid_data[target_col]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = Pipeline([('scaler', StandardScaler()), ('regressor', LinearRegression())])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        return {
            'model': model,
            'metrics': {
                'r2': r2_score(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'n_samples': len(valid_data)
            },
            'feature_importances': dict(zip(available_predictors, 
                                          np.abs(model.named_steps['regressor'].coef_)))
        }
    
    def run_correlation_analysis(self) -> Dict:
        """Run correlation analysis between key variables"""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols)
        
        corr_data = self.filtered_df[key_cols].corr()
        
        significant_corrs = {}
        for i, col1 in enumerate(key_cols):
            for j, col2 in enumerate(key_cols):
                if i < j:
                    valid_data = self.filtered_df[[col1, col2]].dropna()
                    if len(valid_data) > 5:
                        r, p = pearsonr(valid_data[col1], valid_data[col2])
                        if p < 0.05:
                            significant_corrs[f'{col1}_vs_{col2}'] = {'r': r, 'p': p, 'n': len(valid_data)}
        
        return {'correlation_matrix': corr_data.to_dict(), 'significant_correlations': significant_corrs}

# ==============================================================================
# VISUALIZATION COMPONENTS
# ==============================================================================

class PopulationVisualizer(BaseVisualizer):
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        super().__init__(config)
        self.metrics_df = metrics_df
        self.filtered_df = (metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold] 
                           if 'mot_noise' in metrics_df.columns else metrics_df)

    def _create_age_vs_metric_plot(self, metric_suffix: str, ylabel: str, title_suffix: str, 
                                  ylim: Tuple[float, float] = None) -> Path:
        """Generic function to create age vs metric plots"""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate shared axis limits
        metric_cols = [f'{trial}_{metric_suffix}_{condition}_const' for trial in trials for condition in conditions]
        available_cols = [col for col in metric_cols if col in self.filtered_df.columns]
        
        if available_cols and not ylim:
            all_data = []
            for col in available_cols:
                data = self.filtered_df[col].dropna()
                if not data.empty:
                    all_data.extend(data.values)
            
            if all_data:
                y_min, y_max = min(all_data), max(all_data)
                y_range = y_max - y_min
                y_padding = y_range * 0.05
                ylim = [y_min - y_padding, y_max + y_padding]
        
        # Calculate age limits
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min, age_max = age_data.min(), age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_{metric_suffix}_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel(ylabel)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                
                # Set consistent axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
                if ylim:
                    ax.set_ylim(ylim)
        
        plt.suptitle(f'Age vs {title_suffix} by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'age_vs_{metric_suffix}.png'
        return self.save_figure(fig, filename, 'population')

    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('sr', 'Success Rate', 'Success Rates', (-0.05, 1.05))
    
    def plot_age_vs_mean_stride_length(self) -> Path:
        """Plot age vs mean stride length by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('msl', 'Mean Stride Length', 'Mean Stride Length')
    
    def plot_age_vs_stride_variability(self) -> Path:
        """Plot age vs stride length variability by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('sd', 'Stride Length Variability (SD)', 'Stride Length Variability')

    def plot_correlation_matrix(self) -> Path:
        """Plot correlation matrix of key variables"""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols[:6])
        
        if len(key_cols) > 1:
            fig = plt.figure(figsize=(12, 10))
            corr_matrix = self.filtered_df[key_cols].corr()
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                       square=True, linewidths=0.5, fmt='.2f')
            
            plt.title('Correlation Matrix of Key Variables', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            return self.save_figure(fig, 'correlation_matrix.png', 'population')

    def plot_trial_comparison(self) -> Path:
        """Plot comparison across trial types"""
        trial_cols = []
        for trial in ['vis1', 'invis', 'vis2']:
            for condition in ['max', 'min']:
                col = f'{trial}_sr_{condition}_const'
                if col in self.filtered_df.columns:
                    trial_cols.append(col)
        
        if len(trial_cols) < 2:
            return None
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        
        plot_data, labels = [], []
        for col in trial_cols:
            data = self.filtered_df[col].dropna()
            if len(data) > 0:
                plot_data.append(data)
                parts = col.split('_')
                trial_name, condition_name = parts[0].upper(), parts[2].capitalize()
                labels.append(f'{trial_name}\n{condition_name}')
        
        if plot_data:
            ax.boxplot(plot_data, tick_labels=labels)
            ax.set_ylabel('Success Rate')
            ax.set_title('Success Rate Distribution by Trial Type and Condition')
            ax.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        return self.save_figure(fig, 'trial_comparison.png', 'population')

    def _create_anova_subplot(self, metric_suffix: str, ylabel: str) -> Tuple[List, List]:
        """Helper to create ANOVA subplot data"""
        cols = [col for col in self.filtered_df.columns if f'_{metric_suffix}_' in col and '_const' in col]
        if not cols:
            return [], []
        
        plot_data, trial_types, conditions = [], [], []
        for col in cols:
            parts = col.split('_')
            if len(parts) >= 3:
                trial_type, condition = parts[0], parts[2]
                data = self.filtered_df[col].dropna()
                for value in data:
                    plot_data.append(value)
                    trial_types.append(trial_type.upper())
                    conditions.append(condition.capitalize())
        
        return [plot_data, trial_types, conditions], ylabel

    def plot_anova_results(self) -> List[Path]:
        """Create comprehensive plots for ANOVA results"""
        plot_paths = []
        
        for metric_suffix, ylabel, title in [('sr', 'Success Rate', 'Success Rate'), 
                                           ('msl', 'Mean Stride Length', 'Mean Stride Length'),
                                           ('sd', 'Stride Length Variability (SD)', 'Stride Variability')]:
            data_info, ylabel = self._create_anova_subplot(metric_suffix, ylabel)
            if not data_info[0]:
                continue
            
            plot_data, trial_types, conditions = data_info
            df_plot = pd.DataFrame({
                f'{metric_suffix}_value': plot_data, 'trial_type': trial_types, 'condition': conditions
            })
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
            
            # Plot by trial type
            trial_order = ['VIS1', 'INVIS', 'VIS2']
            trial_data = [df_plot[df_plot['trial_type'] == t][f'{metric_suffix}_value'].values 
                         for t in trial_order if t in df_plot['trial_type'].values]
            trial_labels = [t for t in trial_order if t in df_plot['trial_type'].values]
            
            if trial_data:
                bp1 = ax1.boxplot(trial_data, tick_labels=trial_labels, patch_artist=True)
                colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
                for patch, color in zip(bp1['boxes'], colors[:len(bp1['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
            
            ax1.set_ylabel(ylabel)
            ax1.set_title(f'{title} by Trial Type')
            ax1.grid(True, alpha=0.3)
            
            # Plot by condition
            condition_order = ['Max', 'Min']
            condition_data = [df_plot[df_plot['condition'] == c][f'{metric_suffix}_value'].values 
                             for c in condition_order if c in df_plot['condition'].values]
            condition_labels = [c for c in condition_order if c in df_plot['condition'].values]
            
            if condition_data:
                bp2 = ax2.boxplot(condition_data, tick_labels=condition_labels, patch_artist=True)
                colors = [self.colors['success'], self.colors['warning']]
                for patch, color in zip(bp2['boxes'], colors[:len(bp2['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
            
            ax2.set_ylabel(ylabel)
            ax2.set_title(f'{title} by Target Condition')
            ax2.grid(True, alpha=0.3)
            
            plt.suptitle(f'{title} ANOVA Results', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            plot_path = self.save_figure(fig, f'{metric_suffix}_anova.png', 'population')
            if plot_path:
                plot_paths.append(plot_path)
        
        return plot_paths

    def plot_age_stratified_results(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """Create comprehensive visualization of age-stratified ANOVA results"""
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'trial_type_effect' in results]
        
        if not valid_groups:
            print("No valid age groups found for visualization")
            return None
        
        # Set up figure with 5 rows
        n_groups = len(valid_groups)
        fig = plt.figure(figsize=(20, 16))
        gs = fig.add_gridspec(5, max(n_groups, 3), hspace=0.5, wspace=0.3, 
                             height_ratios=[1, 1, 1.3, 1.2, 1])
        
        trial_colors = {'vis1': '#1f77b4', 'invis': '#ff7f0e', 'vis2': '#2ca02c'}
        
        # Row 1: Mean success rates by age group and trial type
        for i, group in enumerate(valid_groups):
            ax = fig.add_subplot(gs[0, i])
            group_data = age_results['group_analyses'][group]
            
            if 'mean_success_rates' in group_data:
                trials = list(group_data['mean_success_rates'].keys())
                means = list(group_data['mean_success_rates'].values())
                colors = [trial_colors.get(trial, 'gray') for trial in trials]
                
                bars = ax.bar(trials, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
                
                for bar, mean in zip(bars, means):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{mean:.2f}', ha='center', va='bottom', fontweight='bold')
                
                ax.set_ylim(0, 1)
                ax.set_ylabel('Mean Success Rate')
                ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})')
                ax.grid(True, alpha=0.3)
                
                age_info = group_data.get('descriptive_stats', {}).get('age_info', {})
                if 'mean_age' in age_info:
                    ax.text(0.5, 0.95, f'Mean age: {age_info["mean_age"]:.1f}y', 
                           transform=ax.transAxes, ha='center', va='top', 
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
        
        # Row 2: Effect sizes comparison
        ax_effect = fig.add_subplot(gs[1, :n_groups])
        effect_sizes, group_names, significance = [], [], []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                group_names.append(group.title())
                significance.append(group_data['trial_type_effect']['significant'])
        
        if effect_sizes:
            colors = ['green' if sig else 'red' for sig in significance]
            bars = ax_effect.bar(group_names, effect_sizes, color=colors, alpha=0.7, edgecolor='black')
            
            for i, (bar, sig, eta2) in enumerate(zip(bars, significance, effect_sizes)):
                height = bar.get_height()
                sig_text = '***' if sig else 'ns'
                ax_effect.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                              f'{eta2:.3f}\n{sig_text}', ha='center', va='bottom', fontweight='bold')
            
            ax_effect.set_ylabel('Effect Size (η²)')
            ax_effect.set_title('Trial Type Effect Sizes by Age Group')
            ax_effect.grid(True, alpha=0.3)
            
            # Add effect size interpretation lines
            for y, label, color in [(0.01, 'Small (0.01)', 'gray'), (0.06, 'Medium (0.06)', 'orange'), (0.14, 'Large (0.14)', 'red')]:
                ax_effect.axhline(y=y, color=color, linestyle='--', alpha=0.5, label=label)
            ax_effect.legend(loc='upper right')
        
        # Row 3: F-statistics comparison
        ax_f = fig.add_subplot(gs[2, :n_groups])
        f_stats, p_values = [], []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                f_stats.append(group_data['trial_type_effect']['F'])
                p_values.append(group_data['trial_type_effect']['p_value'])
        
        if f_stats:
            colors = ['green' if p < 0.05 else 'red' for p in p_values]
            bars = ax_f.bar(group_names, f_stats, color=colors, alpha=0.7, edgecolor='black')
            
            for i, (bar, f_val, p_val) in enumerate(zip(bars, f_stats, p_values)):
                height = bar.get_height()
                p_text = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'
                text_offset = min(0.2, max(f_stats) * 0.05) if f_stats else 0.2
                ax_f.text(bar.get_x() + bar.get_width()/2., height + text_offset,
                         f'F={f_val:.1f}\n{p_text}', ha='center', va='bottom', fontweight='bold')
            
            ax_f.set_ylabel('F-statistic')
            ax_f.set_title('Trial Type F-statistics by Age Group')
            ax_f.grid(True, alpha=0.3)
            ax_f.set_ylim(0, max(f_stats) * 1.3 if f_stats else 1)
        
        # Row 4: Post-hoc Comparisons
        ax_posthoc = fig.add_subplot(gs[3, :])
        
        all_comparison_names = set()
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                all_comparison_names.update(group_data['posthoc_comparisons'].keys())
        
        comparison_names = sorted(list(all_comparison_names))
        comparison_labels = [name.replace('_vs_', ' vs ').upper() for name in comparison_names]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c'][:len(comparison_names)]
        
        if comparison_names:
            x_positions = np.arange(len(valid_groups))
            bar_width = 0.25
            
            for i, (comp_name, comp_label, color) in enumerate(zip(comparison_names, comparison_labels, colors)):
                effect_sizes, significances, p_values = [], [], []
                
                for group in valid_groups:
                    group_data = age_results['group_analyses'][group]
                    if ('posthoc_comparisons' in group_data and 
                        comp_name in group_data['posthoc_comparisons']):
                        comp_data = group_data['posthoc_comparisons'][comp_name]
                        effect_sizes.append(comp_data['cohens_d'])
                        significances.append(comp_data['significant'])
                        p_values.append(comp_data['p_corrected'])
                    else:
                        effect_sizes.append(0.0)
                        significances.append(False)
                        p_values.append(1.0)
                
                bars = ax_posthoc.bar(x_positions + i * bar_width, 
                                     [abs(es) for es in effect_sizes], 
                                     bar_width, label=comp_label, color=color, alpha=0.7)
                
                for j, (bar, es, sig, p_val) in enumerate(zip(bars, effect_sizes, significances, p_values)):
                    height = bar.get_height()
                    
                    if sig:
                        sig_symbol = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else "*")
                        text_color, text_weight = 'black', 'bold'
                    else:
                        sig_symbol = "ns"
                        text_color, text_weight = 'gray', 'normal'
                    
                    if height > 0.01 or es != 0:
                        ax_posthoc.text(bar.get_x() + bar.get_width()/2., max(height + 0.02, 0.05),
                                       f'd={es:.2f}\n{sig_symbol}', 
                                       ha='center', va='bottom', fontsize=9, 
                                       color=text_color, fontweight=text_weight)
                    else:
                        ax_posthoc.text(bar.get_x() + bar.get_width()/2., 0.05,
                                       f'No data', 
                                       ha='center', va='bottom', fontsize=8, 
                                       color='red', style='italic')
            
            ax_posthoc.set_xlabel('Age Groups')
            ax_posthoc.set_ylabel('Effect Size (|Cohen\'s d|)')
            ax_posthoc.set_title('Post-Hoc Pairwise Comparisons: Effect Sizes by Age Group')
            ax_posthoc.set_xticks(x_positions + bar_width)
            ax_posthoc.set_xticklabels([g.title() for g in valid_groups])
            ax_posthoc.legend(loc='upper right')
            ax_posthoc.grid(True, alpha=0.3)
            
            # Add effect size interpretation lines
            for y, label, color in [(0.2, 'Small (0.2)', 'gray'), (0.5, 'Medium (0.5)', 'orange'), (0.8, 'Large (0.8)', 'red')]:
                ax_posthoc.axhline(y=y, color=color, linestyle='--', alpha=0.5)
            
            all_effects = []
            for group in valid_groups:
                group_data = age_results['group_analyses'][group]
                if 'posthoc_comparisons' in group_data:
                    for comp_data in group_data['posthoc_comparisons'].values():
                        all_effects.append(abs(comp_data['cohens_d']))
            
            max_effect = max(all_effects) if all_effects else 0.5
            ax_posthoc.set_ylim(0, max(max_effect * 1.4, 0.3))
        
        # Row 5: Summary table
        ax_table = fig.add_subplot(gs[4, :])
        ax_table.axis('off')
        
        table_data = []
        headers = ['Age Group', 'N', 'Mean Age', 'F-stat', 'p-value', 'η²', 'Significant', 'Key Comparisons']
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data and 'descriptive_stats' in group_data:
                trial_effect = group_data['trial_type_effect']
                desc_stats = group_data['descriptive_stats']
                
                eta2 = trial_effect['effect_size_eta2']
                interpretation = ('Negligible' if eta2 < 0.01 else 
                                'Small' if eta2 < 0.06 else 
                                'Medium' if eta2 < 0.14 else 'Large')
                
                posthoc_summary = "No comparisons"
                if 'posthoc_comparisons' in group_data:
                    posthoc = group_data['posthoc_comparisons']
                    comparison_summaries = []
                    
                    for comp_name, comp_data in posthoc.items():
                        clean_name = comp_name.replace('_vs_', '>')
                        d_val = comp_data['cohens_d']
                        sig_marker = "*" if comp_data['significant'] else "ns"
                        comparison_summaries.append(f"{clean_name}(d={d_val:.2f},{sig_marker})")
                    
                    posthoc_summary = '; '.join(comparison_summaries[:2]) if comparison_summaries else "No comparisons calculated"
                
                row = [
                    group.title(), str(desc_stats['n_subjects']), f"{desc_stats['age_info']['mean_age']:.1f}y",
                    f"{trial_effect['F']:.2f}", 
                    f"{trial_effect['p_value']:.3f}" if trial_effect['p_value'] >= 0.001 else "<0.001",
                    f"{eta2:.3f}", "Yes" if trial_effect['significant'] else "No", posthoc_summary
                ]
                table_data.append(row)
        
        if table_data:
            table = ax_table.table(cellText=table_data, colLabels=headers, cellLoc='center',
                                  loc='center', bbox=[0, 0, 1, 1])
            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1, 2)
            
            # Color code significance
            for i in range(len(table_data)):
                sig_color = '#90EE90' if table_data[i][6] == "Yes" else '#FFB6C1'
                table[(i+1, 6)].set_facecolor(sig_color)
                
                posthoc_text = table_data[i][7]
                if "No comparisons" in posthoc_text:
                    posthoc_color = '#FFE4B5'
                elif "*" in posthoc_text:
                    posthoc_color = '#E0FFE0'
                elif "ns" in posthoc_text:
                    posthoc_color = '#F0F8FF'
                else:
                    posthoc_color = '#FFFFFF'
                table[(i+1, 7)].set_facecolor(posthoc_color)
            
            # Style header
            for j in range(len(headers)):
                table[(0, j)].set_facecolor('#4CAF50')
                table[(0, j)].set_text_props(weight='bold', color='white')
        
        plt.suptitle('Age-Stratified ANOVA Results: Trial Type Effects', fontsize=16, fontweight='bold')
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to: {save_path}")
        else:
            filename = 'age_stratified_anova_results.png'
            save_path = self.save_figure(fig, filename, 'population')
        
        return fig

    def plot_posthoc_heatmap(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """Create heatmap visualization of post-hoc comparisons across age groups"""
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'posthoc_comparisons' in results]
        
        if not valid_groups:
            print("No valid age groups with post-hoc data found")
            return None
        
        all_comparison_names = set()
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                all_comparison_names.update(group_data['posthoc_comparisons'].keys())
        
        comparison_names = sorted(list(all_comparison_names))
        comparison_labels = [name.replace('_vs_', ' vs ').upper() for name in comparison_names]
        
        if not comparison_names:
            print("No comparison names found in data")
            return None
        
        # Create figure with 3 subplots: effect sizes, p-values, and summary
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Post-Hoc Comparison Analysis Across Age Groups', fontsize=16, fontweight='bold')
        
        # Prepare data matrices
        effect_matrix = np.zeros((len(comparison_names), len(valid_groups)))
        p_value_matrix = np.ones((len(comparison_names), len(valid_groups)))
        sig_matrix = np.zeros((len(comparison_names), len(valid_groups)), dtype=bool)
        
        for i, group in enumerate(valid_groups):
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                posthoc = group_data['posthoc_comparisons']
                
                for j, comp_name in enumerate(comparison_names):
                    if comp_name in posthoc:
                        comp_data = posthoc[comp_name]
                        effect_matrix[j, i] = comp_data['cohens_d']
                        p_value_matrix[j, i] = comp_data['p_corrected']
                        sig_matrix[j, i] = comp_data['significant']
        
        # Plot 1: Effect sizes heatmap
        ax1 = axes[0, 0]
        im1 = ax1.imshow(effect_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax1.set_xticks(range(len(valid_groups)))
        ax1.set_xticklabels([g.title() for g in valid_groups])
        ax1.set_yticks(range(len(comparison_labels)))
        ax1.set_yticklabels(comparison_labels)
        ax1.set_title('Effect Sizes (Cohen\'s d)')
        
        # Add text annotations
        for i in range(len(comparison_names)):
            for j in range(len(valid_groups)):
                text = f'{effect_matrix[i, j]:.2f}'
                color = 'white' if abs(effect_matrix[i, j]) > 0.5 else 'black'
                weight = 'bold' if abs(effect_matrix[i, j]) > 0.3 else 'normal'
                ax1.text(j, i, text, ha='center', va='center', color=color, fontweight=weight)
        
        plt.colorbar(im1, ax=ax1, label='Cohen\'s d')
        
        # Plot 2: P-values heatmap
        ax2 = axes[0, 1]
        log_p_matrix = -np.log10(np.maximum(p_value_matrix, 1e-10))
        im2 = ax2.imshow(log_p_matrix, cmap='Reds', aspect='auto')
        ax2.set_xticks(range(len(valid_groups)))
        ax2.set_xticklabels([g.title() for g in valid_groups])
        ax2.set_yticks(range(len(comparison_labels)))
        ax2.set_yticklabels(comparison_labels)
        ax2.set_title('Statistical Significance (-log10(p))')
        
        # Add text annotations
        for i in range(len(comparison_names)):
            for j in range(len(valid_groups)):
                p_val = p_value_matrix[i, j]
                text = ('***' if p_val < 0.001 else '**' if p_val < 0.01 else 
                       '*' if p_val < 0.05 else f'{p_val:.2f}')
                color = 'white' if log_p_matrix[i, j] > 1 else 'black'
                weight = 'bold' if p_val < 0.05 else 'normal'
                ax2.text(j, i, text, ha='center', va='center', color=color, fontweight=weight)
        
        plt.colorbar(im2, ax=ax2, label='-log10(p-value)')
        
        # Plot 3: Effect size vs significance scatter plot
        ax3 = axes[1, 0]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        
        for i, (comp_name, comp_label, color) in enumerate(zip(comparison_names, comparison_labels, colors)):
            group_effects = effect_matrix[i, :]
            group_p_values = p_value_matrix[i, :]
            group_sigs = sig_matrix[i, :]
            
            for j, (effect, p_val, sig) in enumerate(zip(group_effects, group_p_values, group_sigs)):
                marker = 'o' if sig else '^'
                alpha = 0.8 if sig else 0.6
                size = 100 if sig else 80
                edge_color = 'black' if sig else 'gray'
                edge_width = 2 if sig else 1
                
                ax3.scatter(effect, -np.log10(max(p_val, 1e-10)), 
                           color=color, marker=marker, s=size, alpha=alpha,
                           edgecolors=edge_color, linewidths=edge_width,
                           label=f'{comp_label}' if j == 0 else "")
        
        # Add reference lines
        ax3.axhline(y=-np.log10(0.05), color='red', linestyle='--', alpha=0.7, label='p=0.05')
        ax3.axhline(y=-np.log10(0.01), color='orange', linestyle='--', alpha=0.5, label='p=0.01')
        ax3.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        for x, label in [(0.2, 'Small effect'), (-0.2, ''), (0.5, 'Medium effect'), (-0.5, '')]:
            ax3.axvline(x=x, color='gray', linestyle=':', alpha=0.5, label=label if x > 0 else "")
        
        ax3.set_xlabel('Effect Size (Cohen\'s d)')
        ax3.set_ylabel('Statistical Significance (-log10(p))')
        ax3.set_title('Effect Size vs Statistical Significance')
        ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Summary statistics table
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        summary_data = []
        headers = ['Comparison', 'Mean |d|', 'Max |d|', 'N Significant', 'N Groups']
        
        for i, comp_label in enumerate(comparison_labels):
            effects = np.abs(effect_matrix[i, :])
            sigs = sig_matrix[i, :]
            
            summary_data.append([
                comp_label, f'{np.mean(effects):.2f}', f'{np.max(effects):.2f}',
                f'{np.sum(sigs)}/{len(valid_groups)}', str(len(valid_groups))
            ])
        
        table = ax4.table(cellText=summary_data, colLabels=headers, cellLoc='center',
                         loc='center', bbox=[0, 0.3, 1, 0.6])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        
        # Style the table
        for j in range(len(headers)):
            table[(0, j)].set_facecolor('#4CAF50')
            table[(0, j)].set_text_props(weight='bold', color='white')
        
        ax4.set_title('Summary Statistics', pad=20)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Post-hoc heatmap saved to: {save_path}")
        else:
            filename = 'posthoc_heatmap.png'
            save_path = self.save_figure(fig, filename, 'population')
        
        return fig

class IndividualVisualizer(BaseVisualizer):
    def __init__(self, data_manager: 'DataManager', config: AnalysisConfig):
        super().__init__(config)
        self.data_manager = data_manager
    
    def plot_subject_summary(self, subject_id: str) -> Optional[Path]:
        """Plot comprehensive summary for a subject"""
        subject = self.data_manager.subjects.get(subject_id)
        if not subject:
            return None
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Success rates across trials
        self._plot_success_rates(axes[0, 0], subject)
        
        # Plot 2: Motor noise visualization
        self._plot_motor_noise(axes[0, 1], subject)
        
        # Plot 3: Age comparison
        self._plot_age_comparison(axes[1, 0], subject)
        
        # Plot 4: Subject information
        self._plot_subject_info(axes[1, 1], subject)
        
        plt.suptitle(f'Subject {subject_id} Summary', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'subject_summary_{subject_id}.png'
        return self.save_figure(fig, filename, 'individual')
    
    def _plot_success_rates(self, ax, subject):
        """Plot success rates subplot"""
        trial_types = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        x_pos = np.arange(len(trial_types))
        width = 0.35
        
        max_rates, min_rates = [], []
        for trial in trial_types:
            trial_dict = subject.trial_data.get(trial)
            if trial_dict and trial_dict['data'] is not None:
                df = trial_dict['data']
                max_data, _ = self.data_manager._get_period_data(df, 'max')
                min_data, _ = self.data_manager._get_period_data(df, 'min')
                max_rate = max_data['Success'].mean() if max_data is not None else 0
                min_rate = min_data['Success'].mean() if min_data is not None else 0
                max_rates.append(max_rate)
                min_rates.append(min_rate)
            else:
                max_rates.extend([0, 0])
        
        ax.bar(x_pos - width/2, max_rates, width, label='Max Target', alpha=0.8)
        ax.bar(x_pos + width/2, min_rates, width, label='Min Target', alpha=0.8)
        ax.set_xlabel('Trial Type')
        ax.set_ylabel('Success Rate')
        ax.set_title('Success Rates by Trial Type')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([t.upper() for t in trial_types])
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_motor_noise(self, ax, subject):
        """Plot motor noise subplot"""
        pref_dict = subject.trial_data.get('pref')
        if pref_dict and pref_dict['data'] is not None:
            pref_df = pref_dict['data']
            if all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
                right_steps = pref_df['Right step length']
                left_steps = pref_df['Left step length']
                
                ax.plot(right_steps, label='Right Steps', alpha=0.7)
                ax.plot(left_steps, label='Left Steps', alpha=0.7)
                ax.set_xlabel('Stride Number')
                ax.set_ylabel('Step Length')
                ax.set_title('Preference Trial Step Lengths')
                ax.legend()
                ax.grid(True, alpha=0.3)
                return
        
        ax.text(0.5, 0.5, 'No Preference Data', ha='center', va='center',
               transform=ax.transAxes, fontsize=12)
    
    def _plot_age_comparison(self, ax, subject):
        """Plot age comparison subplot"""
        all_ages = [s.age for s in self.data_manager.subjects.values()]
        ax.hist(all_ages, bins=20, alpha=0.7, label='All Subjects')
        ax.axvline(subject.age, color='red', linestyle='--', linewidth=2, 
                   label=f'This Subject (Age: {subject.age:.1f})')
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Frequency')
        ax.set_title('Age Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_subject_info(self, ax, subject):
        """Plot subject information subplot"""
        ax.text(0.5, 0.5, f'Subject ID: {subject.subject_id}\nAge: {subject.age:.1f} years\n'
                f'Session: {subject.metadata.get("Session Date", "Unknown")}', 
                ha='center', va='center', transform=ax.transAxes, fontsize=12,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax.set_title('Subject Information')
        ax.axis('off')

# ==============================================================================
# MAIN ANALYSIS INTERFACE
# ==============================================================================

class MotorLearningAnalysis:
    """Main analysis interface that coordinates all components"""
    
    def __init__(self, config: AnalysisConfig = None):
        self.config = config or AnalysisConfig()
        self.data_manager: Optional[DataManager] = None
        self.metrics_df: Optional[pd.DataFrame] = None
        self.statistical_analyzer: Optional[StatisticalAnalyzer] = None
        self.population_visualizer: Optional[PopulationVisualizer] = None
        self.individual_visualizer: Optional[IndividualVisualizer] = None
        self.results: Optional[Dict] = None
    
    def load_data(self, metadata_path: str, data_root_dir: str, 
                  force_reprocess: bool = False) -> 'MotorLearningAnalysis':
        """Load and process data"""
        self.data_manager = DataManager(metadata_path, data_root_dir, self.config, force_reprocess)
        return self
    
    def filter_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter data based on criteria"""
        if self.data_manager:
            self.data_manager = self.data_manager.filter_subjects(**kwargs)
        return self
    
    def calculate_metrics(self) -> 'MotorLearningAnalysis':
        """Calculate metrics for all subjects"""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        self.metrics_df = self.data_manager.calculate_metrics()
        
        # Initialize analyzers with metrics
        self.statistical_analyzer = StatisticalAnalyzer(self.metrics_df, self.config)
        self.population_visualizer = PopulationVisualizer(self.metrics_df, self.config)
        self.individual_visualizer = IndividualVisualizer(self.data_manager, self.config)
        
        return self
    
    def run_analysis(self, include_visualizations: bool = True) -> 'MotorLearningAnalysis':
        """Run comprehensive analysis and store results"""
        if self.metrics_df is None:
            raise ValueError("Metrics not calculated. Call calculate_metrics() first.")
        
        self.results = {
            'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'n_subjects': len(self.metrics_df),
            'analyses': {}
        }
        
        # Statistical analyses - consolidated error handling
        analyses = [
            ('regression', lambda: self.statistical_analyzer.run_regression_analysis()),
            ('anova_success_rates', lambda: self.statistical_analyzer.run_repeated_measures_anova()),
            ('anova_mean_stride_length', lambda: self.statistical_analyzer.run_repeated_measures_anova_msl()),
            ('anova_stride_variability', lambda: self.statistical_analyzer.run_repeated_measures_anova_sd()),
            ('correlations', lambda: self.statistical_analyzer.run_correlation_analysis()),
            ('age_stratified_anova', lambda: self.statistical_analyzer.run_age_stratified_anova())
        ]
        
        for name, func in analyses:
            try:
                self.results['analyses'][name] = func()
            except Exception as e:
                self.results['analyses'][name] = {'error': str(e)}
        
        # Visualizations
        if include_visualizations and self.population_visualizer:
            self.results['visualizations'] = {'population': [], 'individual': []}
            
            # Population plots - consolidated error handling
            plot_functions = [
                self.population_visualizer.plot_age_vs_success_rates,
                self.population_visualizer.plot_age_vs_mean_stride_length,
                self.population_visualizer.plot_age_vs_stride_variability,
                self.population_visualizer.plot_correlation_matrix,
                self.population_visualizer.plot_trial_comparison
            ]
            
            for plot_func in plot_functions:
                try:
                    plot_path = plot_func()
                    if plot_path:
                        self.results['visualizations']['population'].append(str(plot_path))
                except Exception as e:
                    print(f"Failed to create plot {plot_func.__name__}: {e}")
            
            # ANOVA-specific plots
            try:
                anova_plots = self.population_visualizer.plot_anova_results()
                for plot_path in anova_plots:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create ANOVA plots: {e}")
            
            # Age-stratified ANOVA plots
            try:
                if 'age_stratified_anova' in self.results['analyses'] and 'error' not in self.results['analyses']['age_stratified_anova']:
                    age_results = self.results['analyses']['age_stratified_anova']
                    
                    # Create age-stratified results plot
                    age_plot = self.population_visualizer.plot_age_stratified_results(age_results)
                    if age_plot:
                        plot_path = self.population_visualizer.save_figure(age_plot, 'age_stratified_anova_results.png', 'population')
                        self.results['visualizations']['population'].append(str(plot_path))
                    
                    # Create post-hoc heatmap
                    heatmap_plot = self.population_visualizer.plot_posthoc_heatmap(age_results)
                    if heatmap_plot:
                        plot_path = self.population_visualizer.save_figure(heatmap_plot, 'posthoc_heatmap.png', 'population')
                        self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create age-stratified ANOVA plots: {e}")
        
        # Generate report
        self._generate_report()
        return self
    
    def get_results(self) -> Dict:
        """Get analysis results"""
        return self.results
    
    def _generate_report(self):
        """Generate analysis report"""
        if not self.results:
            return
        
        report_path = self.config.reports_dir / f"analysis_report_{self.results['timestamp']}.json"
        
        def make_serializable(obj):
            if isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, Path):
                return str(obj)
            elif pd.isna(obj):
                return None
            elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
                return str(type(obj).__name__)
            return obj
        
        serializable_results = json.loads(json.dumps(self.results, default=make_serializable))
        
        with open(report_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"Report saved to: {report_path}")

# ==============================================================================
# CONVENIENCE FUNCTIONS
# ==============================================================================

def run_motor_learning_analysis(metadata_path: str, data_root_dir: str,
                               output_dir: str = 'motor_learning_output',
                               required_trials: List[str] = None) -> MotorLearningAnalysis:
    """Convenience function to run complete analysis"""
    config = AnalysisConfig(base_output_dir=Path(output_dir))
    
    return (MotorLearningAnalysis(config)
            .load_data(metadata_path, data_root_dir)
            .filter_data(required_trial_types=required_trials or ['vis1', 'invis', 'vis2'])
            .calculate_metrics()
            .run_analysis())

def main():
    """Example usage"""
    data_root_dir = 'muh_data/'
    metadata_path = 'muh_metadata.csv'
    
    print("Motor Learning Analysis Pipeline - Optimized Version")
    print("=" * 60)
    
    # Quick analysis
    analysis = run_motor_learning_analysis(metadata_path, data_root_dir)
    
    # Access components
    data_manager = analysis.data_manager
    metrics_df = analysis.metrics_df
    stats = analysis.statistical_analyzer
    
    # Run specific analyses
    regression_results = stats.run_regression_analysis()
    anova_results = stats.run_repeated_measures_anova()
    
    # Print summary
    print(f"\nDataset Summary:")
    print(f"  Total subjects: {len(metrics_df)}")
    print(f"  Age range: {metrics_df['age'].min():.1f} - {metrics_df['age'].max():.1f} years")
    
    print(f"\nRegression R²: {regression_results['metrics']['r2']:.3f}")
    print(f"ANOVA trial effect p-value: {anova_results.get('trial_type_effect', {}).get('p_value', 'N/A')}")
    
    print(f"\nAll outputs saved to: {analysis.config.base_output_dir}")
    return analysis

if __name__ == "__main__":
    main()

Motor Learning Analysis Pipeline - Optimized Version
✓ Loaded 110 subjects from cache
📊 Calculating metrics for 66 subjects...
✓ Successfully calculated metrics for 66 subjects


C:\Users\castle\AppData\Local\anaconda3\Lib\site-packages\pingouin\distribution.py:507: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  data.groupby(level=1, axis=1, observed=True, group_keys=False)
C:\Users\castle\AppData\Local\anaconda3\Lib\site-packages\pingouin\distribution.py:508: FutureWarning: DataFrameGroupBy.diff with axis=1 is deprecated and will be removed in a future version. Operate on the un-grouped DataFrame instead
  .diff(axis=1)
C:\Users\castle\AppData\Local\anaconda3\Lib\site-packages\pingouin\distribution.py:507: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  data.groupby(level=1, axis=1, observed=True, group_keys=False)
C:\Users\castle\AppData\Local\anaconda3\Lib\site-packages\pingouin\distribution.py:508: FutureWarning: DataFrameGroupBy.diff with axis=1 is deprecated and will be removed in a future version. Operate on the un-grouped DataFram